In [1]:
import pandas as pd
import json
import re

import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [2]:
# 1. Cargar datos
partidos = pd.read_csv('../data/partidos_rankeados.csv')
with open('../data/jugadores_todos_los_paises.json', 'r', encoding='utf-8') as f:
    jugadores_dict = json.load(f)

# 2. Función para limpiar y transformar el valor de mercado
def clean_market_value(value_str):
    if pd.isna(value_str) or not isinstance(value_str, str):
        return 0.0
    # Remover símbolo de euro y espacios
    val = value_str.replace('€', '').strip()
    if 'm' in val:
        return float(val.replace('m', ''))
    elif 'k' in val:
        return float(val.replace('k', '')) / 1000.0
    return 0.0

In [3]:
# 3. Calcular valor total por país
valores_paises = {}
for pais, lista_jugadores in jugadores_dict.items():
    total_val = sum(clean_market_value(j.get('Market Value', '0')) for j in lista_jugadores)
    valores_paises[pais] = total_val

# 4. Mapear valores a la matriz de partidos
partidos['home_value'] = partidos['home_team'].map(valores_paises).fillna(0.0)
partidos['away_value'] = partidos['away_team'].map(valores_paises).fillna(0.0)

# 5. Ingeniería de Características (Features del Partido)
matriz_partidos = pd.DataFrame()
matriz_partidos['id_partido'] = partidos['id']
matriz_partidos['home_team'] = partidos['home_team']
matriz_partidos['away_team'] = partidos['away_team']

#Features de Fecha
partidos['date'] = pd.to_datetime(partidos['date'])
# OPCIÓN A: Día en formato texto (ej: 'Thursday', 'Friday')
partidos['day_of_week_name'] = partidos['date'].dt.day_name()
# OPCIÓN B: Día en formato numérico (Lunes = 0, Domingo = 6)
matriz_partidos['day_of_week_num'] = partidos['date'].dt.dayofweek
# Crea una columna que vale 1 si es Sábado (5) o Domingo (6), y 0 si es día de semana
matriz_partidos['is_weekend'] = partidos['date'].dt.dayofweek.isin([5, 6]).astype(int)

# Features numéricas puras
matriz_partidos['ELO_match_rating'] = partidos['match_rating']
matriz_partidos['ELO_diff'] = (partidos['home_elo_rating'] - partidos['away_elo_rating']).abs()
matriz_partidos['home_value'] = partidos['home_value']
matriz_partidos['away_value'] = partidos['away_value']
matriz_partidos['match_value'] = partidos['home_value'] + partidos['away_value']
# 1. Aplicar logaritmo natural para achatar la diferencia entre ricos y pobres
#matriz_partidos['match_value_log'] = np.log1p(matriz_partidos['match_value'])

# 2. Ahora sí, aplicar MinMaxScaler a la columna con logaritmo
scaler = MinMaxScaler()
#matriz_partidos['match_value_scaled'] = scaler.fit_transform(matriz_partidos[['match_value_log']])
matriz_partidos['match_value_diff'] = (partidos['home_value'] - partidos['away_value']).abs()
# 1. Aplicar logaritmo para suavizar las diferencias extremas
#matriz_partidos['match_value_diff_log'] = np.log1p(matriz_partidos['match_value_diff'])

# 2. Escalar al rango [0, 1] con tu scaler previamente definido
#matriz_partidos['match_value_diff_scaled'] = scaler.fit_transform(matriz_partidos[['match_value_diff_log']])
matriz_partidos['match_hora_utc'] = pd.to_datetime(partidos['time_utc'], format='%H:%M:%S').dt.hour

# Guardar o inspeccionar la matriz resultante
matriz_partidos

,id_partido,home_team,away_team,day_of_week_num,is_weekend,ELO_match_rating,ELO_diff,home_value,away_value,match_value,match_value_diff,match_hora_utc
0,2,Mexico,South Africa,3,0,3382,334,85.600,52.700,138.300,32.900,19
1,1,Korea Republic,Czechia,4,0,3478,26,143.400,196.425,339.825,53.025,2
2,7,Canada,Bosnia-Herzegovina,4,0,3378,190,222.450,140.400,362.850,82.050,19
3,20,USA,Paraguay,5,1,3554,112,356.700,139.000,495.700,217.700,1
4,9,Qatar,Switzerland,5,1,3314,464,20.475,317.600,338.075,297.125,19
...,...,...,...,...,...,...,...,...,...,...,...,...
67,72,Croatia,Ghana,5,1,3435,425,356.300,295.175,651.475,61.125,21
68,65,Congo DR,Uzbekistan,5,1,2934,520,147.250,78.725,225.975,68.525,23
69,66,Colombia,Portugal,5,1,3959,9,303.950,965.000,1268.950,661.050,23
70,57,Algeria,Austria,6,1,3570,84,227.800,259.700,487.500,31.900,2


In [4]:
matriz_partidos = matriz_partidos.sort_values(by='id_partido', ascending=True)
matriz_partidos.to_csv('../data/matriz_partidos.csv', index=False, encoding="utf-8")

In [6]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Inicializamos el scaler
scaler = MinMaxScaler()

# 1. Escalar variables de ELO directamente
matriz_partidos[['ELO_match_rating_scaled', 'ELO_diff_scaled']] = scaler.fit_transform(
    matriz_partidos[['ELO_match_rating', 'ELO_diff']]
)

# 2. Aplicar logaritmo natural para los valores monetarios (achata los extremos)
matriz_partidos['match_value_log'] = np.log1p(matriz_partidos['match_value'])
matriz_partidos['match_value_diff_log'] = np.log1p(matriz_partidos['match_value_diff'])

# 3. Escalar los valores monetarios logarítmicos
matriz_partidos[['match_value_scaled', 'match_value_diff_scaled']] = scaler.fit_transform(
    matriz_partidos[['match_value_log', 'match_value_diff_log']]
)

# Ya podemos descartar las columnas logarítmicas temporales si queremos limpiar
matriz_partidos = matriz_partidos.drop(columns=['match_value_log', 'match_value_diff_log'])

matriz_partidos

,id_partido,home_team,away_team,day_of_week_num,is_weekend,ELO_match_rating,ELO_diff,home_value,away_value,match_value,match_value_diff,match_hora_utc,ELO_match_rating_scaled,ELO_diff_scaled,match_value_scaled,match_value_diff_scaled
1,1,Korea Republic,Czechia,4,0,3478,26,143.400,196.425,339.825,53.025,2,0.484417,0.022135,0.469576,0.448777
0,2,Mexico,South Africa,3,0,3382,334,85.600,52.700,138.300,32.900,19,0.398931,0.423177,0.205817,0.370770
24,3,Czechia,South Africa,3,0,3250,202,196.425,52.700,249.125,143.725,16,0.281389,0.251302,0.378366,0.613715
27,4,Mexico,Korea Republic,4,0,3610,106,85.600,143.400,229.000,57.800,1,0.601959,0.126302,0.353639,0.462953
53,5,South Africa,Korea Republic,3,0,3276,228,52.700,143.400,196.100,90.700,1,0.304541,0.285156,0.308133,0.537336
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21,68,England,Croatia,2,0,3950,90,1315.000,356.300,1671.300,958.700,20,0.904720,0.105469,0.938463,0.930370
66,69,Panama,England,5,1,3757,283,32.850,1315.000,1347.850,1282.150,21,0.732858,0.356771,0.875099,0.978988
22,70,Ghana,Panama,2,0,3242,232,295.175,32.850,328.025,262.325,23,0.274265,0.290365,0.459189,0.713904
45,71,England,Ghana,1,0,3525,515,1315.000,295.175,1610.175,1019.825,20,0.526269,0.658854,0.927486,0.940706


In [7]:
matriz_partidos.to_csv('../data/matriz_partidos_scaled.csv', index=False, encoding="utf-8")